# IND320 — Project Part 2

## 1. Verify the Spark–Cassandra connection

Run the cells below in order using the project’s Python 3.12 `.venv` kernel in VS Code. Docker Desktop and the `cassandra` container must be running.

This check writes three sample rows to a dedicated `ind320_setup.spark_connection_test` table and reads them back through Spark. Repeating the check updates the same three primary keys. The sample rows are setup data, not API data.

The connector and port follow the course [Cassandra notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/2_Databases/3_Cassandra.ipynb) and [Spark notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/2_Databases/4_Spark.ipynb).

### 1.1 Check Python and select Java 17
Restart the notebook kernel before this cell if a Spark session is already running.

In [4]:
import os
import platform
import subprocess
import sys
from pathlib import Path

# The course setup uses Python 3.12 and Spark 3.5.1 with Java 17.
assert sys.version_info[:2] == (3, 12), "Select the project's Python 3.12 .venv kernel."
if platform.system() == "Darwin":
    os.environ["JAVA_HOME"] = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "17"], text=True
    ).strip()

# Spark's Python workers must use the same environment as this notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Support launching the notebook from either the project root or notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "requirements.txt").exists(), "Open this notebook inside the project."
ivy_cache = project_root / "no_sync" / "spark_ivy"
ivy_cache.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Java home:", os.environ.get("JAVA_HOME", "not configured"))

Python: 3.12.14
Interpreter: /Users/agob/Projects/Streamlit_dashboard/.venv/bin/python
Java home: /Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home


### 1.2 Connect to Cassandra and prepare a test table
The Python driver creates the table structure. Spark will write and read the actual test rows.

In [5]:
from cassandra.cluster import Cluster

# Port 9042 is forwarded from Docker to this computer.
cluster = Cluster(["127.0.0.1"], port=9042, connect_timeout=10)
try:
    session = cluster.connect()
    version = session.execute("SELECT release_version FROM system.local").one()
    print("Cassandra connected:", version.release_version)

    # A single replica is sufficient for this local, single-container setup.
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS ind320_setup
        WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
    """)
    session.execute("""
        CREATE TABLE IF NOT EXISTS ind320_setup.spark_connection_test (
            run_id text,
            id int,
            area text,
            value double,
            PRIMARY KEY (run_id, id)
        )
    """)
    print("Test table ready.")
finally:
    # Release the Python driver's connections after preparing the table.
    cluster.shutdown()

NoHostAvailable: ('Unable to connect to any servers', {'127.0.0.1:9042': ConnectionRefusedError(61, "Tried connecting to [('127.0.0.1', 9042)]. Last error: Connection refused")})

### 1.3 Start Spark with the Cassandra connector
The first run downloads the connector and its Java dependencies. This can take a few minutes. Download messages and a native Hadoop library warning may appear.

In [ ]:
from pyspark.sql import SparkSession

# Match the connector version and configuration used in the course notebook.
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("IND320-Cassandra-Check")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .config("spark.jars.ivy", str(ivy_cache))
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark connected:", spark.version)

### 1.4 Write sample rows using Spark

In [ ]:
# Fixed primary keys make this check safe to rerun without duplicate rows.
run_id = "setup-check"
sample_rows = [(run_id, 1, "NO1", 10.0), (run_id, 2, "NO2", 20.0), (run_id, 3, "NO3", 30.0)]
test_data = spark.createDataFrame(
    sample_rows, schema="run_id string, id int, area string, value double"
)
(
    test_data.write
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="ind320_setup", table="spark_connection_test")
    .mode("append")
    .save()
)
print("Spark wrote 3 sample rows to Cassandra.")

### 1.5 Read the rows back and verify the result

In [ ]:
# Read from Cassandra, rather than checking the original in-memory DataFrame.
from pyspark.sql import functions as F

stored_data = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="ind320_setup", table="spark_connection_test")
    .load()
    .filter(F.col("run_id") == run_id)
    .select("id", "area", "value")
    .orderBy("id")
)
actual = [(row.id, row.area, row.value) for row in stored_data.collect()]
expected = [(row[1], row[2], row[3]) for row in sample_rows]
assert actual == expected, f"Read-back mismatch: {actual}"
for row in actual:
    print(row)
print("PASS: Spark wrote to Cassandra and read back all 3 rows correctly.")

### 1.6 Stop the test session
This stops Spark. Cassandra remains running in Docker, and the test rows remain in the dedicated test table. To repeat the test, run the notebook from the top.

In [ ]:
# Free the local Spark resources when the check is finished.
spark.stop()
print("Spark stopped. Cassandra is still running in Docker.")

## 2. Verify the MongoDB connection and document storage

This section follows the [IND320 MongoDB notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/2_Databases/5_MongoDB.ipynb): use PyMongo, ping the cluster, select a database and collection, insert a document, and read it back.

For this project, credentials are read from the local `.streamlit/secrets.toml` file using the same `[mongo]` / `uri` structure as Streamlit Cloud. This replaces the course notebook's `No_sync/MongoDB` credential file. Never paste credentials into notebook cells or outputs.

**Run section 2 on its own, from top to bottom.** It does not need Spark or the Cassandra container. Keep the project’s Python 3.12 `.venv` kernel selected.

The test writes at most one small document in `ind320_setup.connection_test`. A fixed `_id` prevents duplicate test documents. No API data or sample datasets are uploaded. The BSON-size check measures only this document, not database or index overhead. Larger imports must have a separate storage and transfer budget check against the Atlas Free limits.

### 2.1 Read Streamlit Secrets and ping MongoDB

In [6]:
from pathlib import Path
import tomllib

from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Locate the project's ignored secrets file from either working directory.
mongo_project_root = Path.cwd()
if mongo_project_root.name == "notebooks":
    mongo_project_root = mongo_project_root.parent
mongo_secrets_path = mongo_project_root / ".streamlit" / "secrets.toml"

mongo_client = None
try:
    with mongo_secrets_path.open("rb") as secrets_file:
        mongo_uri = tomllib.load(secrets_file)["mongo"]["uri"]

    # Follow the course's MongoClient and Stable API setup.
    # Bound the connection pool and timeouts for this small test.
    mongo_client = MongoClient(
        mongo_uri,
        server_api=ServerApi("1"),
        serverSelectionTimeoutMS=10000,
        connectTimeoutMS=10000,
        socketTimeoutMS=10000,
        maxPoolSize=2,
    )
    del mongo_uri  # Do not display the URI in notebook outputs.
    assert mongo_client.admin.command("ping")["ok"] == 1
    print("PASS: Connected to MongoDB Atlas. Ping wrote no documents.")
except Exception:
    if mongo_client is not None:
        mongo_client.close()
    # Avoid printing driver errors that could contain connection details.
    raise RuntimeError(
        "MongoDB connection failed. Check the local secrets file, database credentials, "
        "and Atlas IP access list. Connection details are hidden."
    ) from None

PASS: Connected to MongoDB Atlas. Ping wrote no documents.


### 2.2 Insert one small document and read it back
Run this cell after 2.1. It closes the MongoDB client when finished. To repeat the check, rerun 2.1 and then 2.2.

In [7]:
from bson import BSON

# Use a dedicated setup collection and one fixed primary key.
# These are invented test values, not electricity or reservoir observations.
test_document = {
    "_id": "mongodb-connection-check",
    "purpose": "IND320 connection test",
    "value": 1,
}

try:
    # A strict size guard keeps this test payload below 1 KiB.
    document_bytes = len(BSON.encode(test_document))
    assert document_bytes <= 1024, "The setup document exceeds the test size limit."

    mongo_database = mongo_client["ind320_setup"]
    mongo_collection = mongo_database["connection_test"]
    test_filter = {"_id": test_document["_id"]}

    # The course uses insert_one. Check the fixed ID first so reruns add no copies.
    existing_document = mongo_collection.find_one(test_filter)
    if existing_document is None:
        mongo_collection.insert_one(test_document)
        print("Inserted one test document.")
    elif existing_document == test_document:
        print("The test document already exists; no additional document was inserted.")
    else:
        raise ValueError("An existing document uses the test ID; it was not changed.")

    # Fetch the persisted document from MongoDB and compare its full content.
    stored_document = mongo_collection.find_one(test_filter)
    assert stored_document == test_document, "The stored document differs from the input."
    print("Read back:", stored_document)
    print("Test document BSON size:", document_bytes, "bytes (excluding index/database overhead).")
    print("PASS: MongoDB stored and returned the test document correctly.")
except Exception:
    raise RuntimeError(
        "MongoDB write/read verification failed. Check the connection, write permissions, "
        "and any existing test document. Connection details are hidden."
    ) from None
finally:
    # Close the small connection pool after the test; Atlas stays available.
    mongo_client.close()

The test document already exists; no additional document was inserted.
Read back: {'_id': 'mongodb-connection-check', 'purpose': 'IND320 connection test', 'value': 1}
Test document BSON size: 86 bytes (excluding index/database overhead).
PASS: MongoDB stored and returned the test document correctly.


### 2.3 What this check demonstrates

A successful result verifies that Python can authenticate to Atlas and insert and retrieve a document. The test document remains in `ind320_setup.connection_test` and can be viewed in Atlas Data Explorer. Repeating the test reuses it.

This is a setup check only. The assignment still requires inserting the relevant Spark-extracted API data into MongoDB and querying those data from Streamlit. Review Atlas storage usage before any bulk import and avoid duplicate uploads.

## 3. Inspect a small NVE API sample

This section follows the HTTP GET and JSON workflow in the IND320 [API notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/3_APIs/1_APIs.ipynb) and [REST notebook](https://github.com/khliland/IND320/blob/main/D2Dbook/2_Data_sources/3_APIs/2_REST.ipynb). We adapt the URL to NVE and turn the returned JSON into pandas tables. Unlike the course's testing example with `verify=False`, we retain Requests' default TLS certificate verification. We also add timeouts and a response-size guard to control resource use.

Run section 3 independently, from top to bottom. It needs no credentials, Spark session, or database connection.

**Scope:** two small GET requests: the latest published week and the area metadata. We do not download the full historical dataset or write to Cassandra/MongoDB. Each response is limited to 128 KiB of decoded JSON; the complete section has a 256 KiB JSON-body budget. Notebook outputs retain only these small tables. Rerunning the download cell makes two new requests.

Sources: [NVE Swagger/API schema](https://biapi.nve.no/magasinstatistikk/swagger/index.html), [NVE API description](https://api.nve.no/doc/magasinstatistikk/), and [About reservoir statistics](https://www.nve.no/energi/analyser-og-statistikk/om-magasinstatistikken/). NVE data are provided under NLOD, compatible with CC BY 3.0 Norway; credit NVE when using the data.

### 3.1 Fetch the latest week and area descriptions

In [8]:
import json
from datetime import datetime, timezone

import pandas as pd
import requests
from IPython.display import display

NVE_BASE_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/"
NVE_RESPONSE_LIMIT = 128 * 1024
NVE_SMALL_ENDPOINTS = {"HentOffentligDataSisteUke", "HentOmråder"}


def fetch_small_nve_json(endpoint):
    # Restrict this introductory check to the two small, documented endpoints.
    if endpoint not in NVE_SMALL_ENDPOINTS:
        raise ValueError("This sample helper only supports the two small NVE endpoints.")

    payload = bytearray()
    with requests.get(NVE_BASE_URL + endpoint, timeout=(10, 30), stream=True) as response:
        response.raise_for_status()
        # Stop reading if the decoded body exceeds the per-response budget.
        for chunk in response.iter_content(chunk_size=4096):
            if len(payload) + len(chunk) > NVE_RESPONSE_LIMIT:
                raise RuntimeError("NVE response exceeded the 128 KiB sample budget.")
            payload.extend(chunk)

    return json.loads(payload), len(payload)


nve_latest_json, nve_latest_bytes = fetch_small_nve_json("HentOffentligDataSisteUke")
nve_areas_json, nve_areas_bytes = fetch_small_nve_json("HentOmråder")
assert isinstance(nve_latest_json, list) and nve_latest_json, "Expected weekly observations."
assert isinstance(nve_areas_json, list) and nve_areas_json, "Expected a list of area-group objects."

print("Retrieved at (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Weekly observations:", len(nve_latest_json))
print("Latest-week JSON:", nve_latest_bytes, "bytes")
print("Area-metadata JSON:", nve_areas_bytes, "bytes")
print("Total decoded JSON:", nve_latest_bytes + nve_areas_bytes, "bytes (HTTP overhead excluded)")

Retrieved at (UTC): 2026-10-08T18:22:39+00:00
Weekly observations: 9
Latest-week JSON: 2474 bytes
Area-metadata JSON: 1734 bytes
Total decoded JSON: 4208 bytes (HTTP overhead excluded)


### 3.2 Inspect geographic coverage

The area type and number form a joint key. In this API, `EL` identifies the five electricity price areas, `NO` with number 0 identifies Norway as a whole, and `VASS` identifies the reservoir-statistics watercourse regions. For example, `EL` / 1 has the display name `NO 1`; it must not be confused with the `NO` area type.

We use NVE's returned names and geographic descriptions rather than inventing a mapping. The metadata can include areas for which the latest-week endpoint has no observation.

In [9]:
# HentOmråder returns a list whose objects contain land/elspot/vassdrag lists.
area_records = []
for grouped_areas in nve_areas_json:
    for group in ("land", "elspot", "vassdrag"):
        area_records.extend(grouped_areas.get(group) or [])

nve_areas = pd.DataFrame(area_records)
area_key = ["omrType", "omrnr"]
assert not nve_areas.duplicated(area_key).any(), "Duplicate NVE area keys."
nve_areas = nve_areas.sort_values(area_key).reset_index(drop=True)
with pd.option_context("display.max_colwidth", None):
    display(nve_areas[["omrType", "omrnr", "navn", "beskrivelse"]])

,omrType,omrnr,navn,beskrivelse
0,EL,1,NO 1,Sørøst-Norge. Omfatter østlige del av Østlandet fra Buskerud og nordover (bortsett fra den del av Innlandet som ligger vest og nord for Vågåmo).
1,EL,2,NO 2,"Sørvest-Norge. Omfatter mesteparten av Vestfold, Telemark, Agder, Rogaland og sørlige del av Vestland."
2,EL,3,NO 3,"Midt-Norge. Omfatter nordre og vestlige del av Vestland, den del av Innlandet som ligger vest og nord for Vågåmo, Møre og Romsdal og Trøndelag til Tunnsjødal."
3,EL,4,NO 4,Nord-Norge. Omfatter resten av Trøndelag og Nord-Norge.
4,EL,5,NO 5,"Vest-Norge. Omfatter midtre del av Vestland opp til Sognefjorden og Indre Sogn, og vestlig del av Buskerud."
5,NO,0,Norge,Hele landet
6,VASS,1,VASS1,"Sørøst-Norge. Østlandet, Agder og deler av Rogaland."
7,VASS,2,VASS2,Vest-Norge. Resten av Rogaland og Vestland.
8,VASS,3,VASS3,"Midt-Norge. Møre og Romsdal, Trøndelag og sørlige Nordland"
9,VASS,4,VASS4,Nord-Norge. Resten av Nordland og nordover.


### 3.3 Inspect fields, dates, and filling levels

- `dato_Id` is the observation date. Keep it separate from `neste_Publiseringsdato`, the next publication timestamp.
- `iso_aar` and `iso_uke` are ISO year and week. ISO year can differ from calendar year around New Year; preserve the API's ISO fields.
- As in Part 1, treat `0001-01-01T00:00:00` in the publication field as a missing-value marker.
- `fyllingsgrad` is a fraction: multiply by 100 for display as a percentage.
- `fylling_TWh` is stored energy, while `kapasitet_TWh` is capacity.

The API's date strings here contain no UTC offset. We parse them as supplied, without silently assuming they are UTC. The UTC timestamp printed in 3.1 records when we fetched the response, not when the reservoir observation was made.

In [10]:
nve_latest = pd.DataFrame(nve_latest_json)
required_fields = {
    "dato_Id", "omrType", "omrnr", "iso_aar", "iso_uke", "fyllingsgrad",
    "fylling_TWh", "kapasitet_TWh", "neste_Publiseringsdato",
}
assert required_fields.issubset(nve_latest.columns), "The expected NVE schema has changed."
print("Returned fields:", ", ".join(nve_latest.columns))

# Use explicit names to keep observation and publication times separate.
nve_latest["observation_date"] = pd.to_datetime(nve_latest["dato_Id"], errors="raise")
publication_values = nve_latest["neste_Publiseringsdato"].replace(
    "0001-01-01T00:00:00", pd.NA
)
nve_latest["next_publication_date"] = pd.to_datetime(publication_values, errors="raise")
nve_latest["filling_percent"] = nve_latest["fyllingsgrad"] * 100
assert not nve_latest.duplicated(["observation_date", *area_key]).any(), "Duplicate weekly observations."

# Check ISO fields against each observation date, without using calendar year.
iso_dates = nve_latest["observation_date"].dt.isocalendar()
assert (iso_dates["year"] == nve_latest["iso_aar"]).all(), "ISO year mismatch."
assert (iso_dates["week"] == nve_latest["iso_uke"]).all(), "ISO week mismatch."

# Join on BOTH area fields; numbers overlap across area types.
nve_preview = nve_latest.merge(
    nve_areas[[*area_key, "navn"]], on=area_key, how="left", validate="many_to_one"
).sort_values(area_key)
assert nve_preview["navn"].notna().all(), "An observation has no matching area description."
display(nve_preview[[
    "observation_date", "iso_aar", "iso_uke", "omrType", "omrnr", "navn",
    "filling_percent", "fylling_TWh", "kapasitet_TWh", "next_publication_date",
]].round({"filling_percent": 2, "fylling_TWh": 3, "kapasitet_TWh": 3}))

# Report missing coverage explicitly; do not invent zero-valued observations.
observed_keys = set(map(tuple, nve_latest[area_key].to_numpy()))
metadata_keys = set(map(tuple, nve_areas[area_key].to_numpy()))
print("Areas in metadata without a latest-week row:", sorted(metadata_keys - observed_keys))
print("PASS: Small NVE sample fetched; schema, dates and area mapping checked.")
print("Database writes: 0. Full historical downloads: 0.")

Returned fields: dato_Id, omrType, omrnr, iso_aar, iso_uke, fyllingsgrad, kapasitet_TWh, fylling_TWh, neste_Publiseringsdato, fyllingsgrad_forrige_uke, endring_fyllingsgrad


,observation_date,iso_aar,iso_uke,omrType,omrnr,navn,filling_percent,fylling_TWh,kapasitet_TWh,next_publication_date
1,2026-10-04,2026,40,EL,1,NO 1,75.14,4.511,6.003,2026-10-14 13:00:00
0,2026-10-04,2026,40,EL,2,NO 2,51.23,17.439,34.044,2026-10-14 13:00:00
3,2026-10-04,2026,40,EL,3,NO 3,71.87,6.411,8.921,2026-10-14 13:00:00
2,2026-10-04,2026,40,EL,4,NO 4,89.44,18.853,21.079,2026-10-14 13:00:00
4,2026-10-04,2026,40,EL,5,NO 5,70.93,12.335,17.391,2026-10-14 13:00:00
5,2026-10-04,2026,40,NO,0,Norge,67.89,59.358,87.438,2026-10-14 13:00:00
7,2026-10-04,2026,40,VASS,1,VASS1,61.58,22.197,36.044,2026-10-14 13:00:00
8,2026-10-04,2026,40,VASS,2,VASS2,58.74,13.650,23.238,2026-10-14 13:00:00
6,2026-10-04,2026,40,VASS,3,VASS3,84.20,23.705,28.155,2026-10-14 13:00:00


Areas in metadata without a latest-week row: [('VASS', 4)]
PASS: Small NVE sample fetched; schema, dates and area mapping checked.
Database writes: 0. Full historical downloads: 0.


### 3.4 Limits of this sample and the next step

One week is enough to inspect the response structure and area mapping, but not to recreate the historical line plots from Part 1. The Swagger schema inspected for this step documents no date-range parameters for `HentOffentligData` or `HentOffentligDataSisteUke`. Filtering a DataFrame after downloading is not the same as restricting the API download itself. We must account for that when designing the assignment's interval control and caching.

The next step is to plan a bounded historical retrieval and recreate the Part 1 plots from API data. The existing CSV-based app is still the Part 1 implementation; this small API check does not yet replace it.

## AI usage — setup draft

Codex helped create and check this setup notebook, using the course Cassandra, Spark, MongoDB, and API examples together with NVE documentation as references. Update this description as the project develops.

This notebook currently documents the database connection and write/read checks and a small NVE API inspection. The API work, figures, project links, and required 300–500-word work log will be added during the assignment.